# Projet BI - YouTube Statistics
**Module :** Entrepot de Donnees - 2LGL 2025/2026

## Partie A - Exploration des donnees

### 1. Chargement du dataset avec Pandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

from google.colab import files

DATA_FILE = 'Global YouTube Statistics.csv'
if not os.path.exists(DATA_FILE):
    uploaded = files.upload()

df = pd.read_csv(DATA_FILE, encoding='latin1')
print('dataset charge')
print(df.shape)

### 2. Affichage des informations generales

In [ ]:
print('Apercu des 5 premieres lignes :')
display(df.head())

print('\nColonnes du dataset :')
print(df.columns.tolist())

print('\nInfos sur le dataset :')
df.info()

### 3. Analyse des donnees

In [ ]:
print('Statistiques descriptives :')
display(df.describe())

print('\nValeurs manquantes :')
display(df.isnull().sum())

print('\nTop 5 pays :')
display(df['Country'].value_counts().head())

print('\nTop 5 categories :')
display(df['category'].value_counts().head())

### 4. Distribution des variables numeriques et categorielles

In [ ]:
sns.set_style('whitegrid')

plt.figure(figsize=(12, 6))
sns.histplot(df['subscribers'], bins=50, kde=True)
plt.title('Distribution du nombre d abonnes')
plt.xlabel('Abonnes')
plt.ylabel('Nombre de chaines')
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
top_subscribers = df.nlargest(10, 'subscribers')
sns.barplot(x='Youtuber', y='subscribers', data=top_subscribers, palette='viridis')
plt.title('Top 10 YouTubers par abonnes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
top_categories = df['category'].value_counts().nlargest(10)
sns.barplot(x=top_categories.index, y=top_categories.values, palette='coolwarm')
plt.title('Top 10 Categories de chaines YouTube')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
top_countries = df['Country'].value_counts().nlargest(10)
sns.barplot(x=top_countries.index, y=top_countries.values, palette='Spectral')
plt.title('Top 10 Pays avec le plus de YouTubers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['subscribers', 'video views', 'uploads', 'highest_monthly_earnings', 'highest_yearly_earnings']
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Heatmap - correlation entre variables numeriques')
plt.tight_layout()
plt.show()

### 5. Detection des outliers : boxplots sur les colonnes numeriques cles

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y=df['subscribers'])
plt.title('Boxplot - Abonnes')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(y=df['video views'])
plt.title('Boxplot - Vues')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(y=df['highest_yearly_earnings'])
plt.title('Boxplot - Revenus annuels')
plt.show()

### 6. Observations et constats

**Constat 1 :** La plupart des chaines ont peu d abonnes mais quelques chaines comme T-Series et MrBeast ont beaucoup plus que les autres. On voit ca dans l histogramme et les boxplots (beaucoup d outliers).

**Constat 2 :** Les USA et l Inde ont le plus de YouTubers dans le dataset. Entertainment et Music sont les categories les plus frequentes. La heatmap montre une forte correlation entre abonnes et revenus.

**Constat 3 :** Il y a des valeurs manquantes sur Country, category et subscribers_for_last_30_days qu on va corriger dans l ETL.

---
## Partie B - ETL (Extract, Transform, Load)

### B.1 Extract

In [ ]:
print('Extraction :')
print('lignes =', len(df))
print('colonnes =', list(df.columns))
display(df.head(3))

### B.2 Transform

In [ ]:
cols_cat = ['category', 'Country', 'Abbreviation', 'channel_type', 'created_month']
for col in cols_cat:
    if col in df.columns:
        df[col] = df[col].fillna('Inconnu')

cols_num = df.select_dtypes(include=['float64', 'int64']).columns
for col in cols_num:
    df[col] = df[col].fillna(df[col].median())

int_cols = ['video views', 'uploads', 'subscribers_for_last_30_days', 'created_year', 'created_date', 'rank', 'subscribers']
for col in int_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12,'Inconnu':1}
df['created_month_num'] = df['created_month'].map(month_map)

def make_date(row):
    try:
        return pd.to_datetime(f"{int(row['created_year'])}-{int(row['created_month_num'])}-{int(row['created_date'])}")
    except:
        return pd.NaT

df['created_date_full'] = df.apply(make_date, axis=1)
df = df.drop(columns=['created_month_num'])
df = df.drop_duplicates(subset=['Youtuber', 'Title'], keep='first')

print('valeurs manquantes apres nettoyage :')
display(df.isnull().sum())

In [ ]:
dim_chaine = df[['Youtuber', 'Title', 'channel_type']].drop_duplicates().reset_index(drop=True)
dim_chaine.insert(0, 'id_chaine', range(1, len(dim_chaine)+1))
dim_chaine.columns = ['id_chaine', 'youtuber', 'titre', 'type_chaine']

dim_date = df[['created_year', 'created_month', 'created_date']].drop_duplicates().reset_index(drop=True)
dim_date.insert(0, 'id_date', range(1, len(dim_date)+1))
dim_date.columns = ['id_date', 'annee', 'mois', 'jour']

dim_geo = df[['Country', 'Abbreviation']].drop_duplicates().reset_index(drop=True)
dim_geo.insert(0, 'id_geo', range(1, len(dim_geo)+1))
dim_geo.columns = ['id_geo', 'pays', 'code_pays']

dim_categorie = df[['category']].drop_duplicates().reset_index(drop=True)
dim_categorie.insert(0, 'id_categorie', range(1, len(dim_categorie)+1))
dim_categorie.columns = ['id_categorie', 'categorie']

df2 = df.copy()
df2 = df2.merge(dim_chaine, left_on=['Youtuber','Title','channel_type'], right_on=['youtuber','titre','type_chaine'])
df2 = df2.merge(dim_date, left_on=['created_year','created_month','created_date'], right_on=['annee','mois','jour'])
df2 = df2.merge(dim_geo, left_on=['Country','Abbreviation'], right_on=['pays','code_pays'])
df2 = df2.merge(dim_categorie, left_on=['category'], right_on=['categorie'])

fact_performance = pd.DataFrame({
    'id_fait': range(1, len(df2)+1),
    'id_chaine': df2['id_chaine'],
    'id_date': df2['id_date'],
    'id_geo': df2['id_geo'],
    'id_categorie': df2['id_categorie'],
    'abonnes': df2['subscribers'],
    'vues': df2['video views'],
    'uploads': df2['uploads'],
    'revenu_mensuel_max': df2['highest_monthly_earnings'],
    'revenu_annuel_max': df2['highest_yearly_earnings']
})

print('DIM_CHAINE', len(dim_chaine))
print('DIM_DATE', len(dim_date))
print('DIM_GEO', len(dim_geo))
print('DIM_CATEGORIE', len(dim_categorie))
print('FACT_PERFORMANCE', len(fact_performance))
display(fact_performance.head())

### Tableau ETL

| Categorie | Colonne source | Operation | Resultat cible |
|-----------|---------------|-----------|----------------|
| Extract | Global YouTube Statistics.csv | pd.read_csv(encoding='latin1') | DataFrame 995 lignes |
| Extract | colonnes | verification schema | 28 colonnes |
| Transform | category, Country... | fillna('Inconnu') | colonnes categorielles |
| Transform | colonnes numeriques | fillna(mediane) | valeurs manquantes corrigees |
| Transform | created_year, created_month | conversion types + date | created_date_full |
| Transform | Youtuber + Title | drop_duplicates() | donnees dedupliquees |
| Transform | colonnes uniques | extraction + id | 4 tables dimensions |
| Transform | mesures + FK | jointures | FACT_PERFORMANCE |
| Load | DataFrames | to_sql() | youtube_dwh.db |

### B.3 Load

In [ ]:
conn = sqlite3.connect('youtube_dwh.db')
conn.execute('PRAGMA foreign_keys = ON')

ddl = '''
DROP TABLE IF EXISTS FACT_PERFORMANCE;
DROP TABLE IF EXISTS DIM_CHAINE;
DROP TABLE IF EXISTS DIM_DATE;
DROP TABLE IF EXISTS DIM_GEO;
DROP TABLE IF EXISTS DIM_CATEGORIE;

CREATE TABLE DIM_CHAINE (
    id_chaine INTEGER PRIMARY KEY,
    youtuber TEXT NOT NULL,
    titre TEXT NOT NULL,
    type_chaine TEXT NOT NULL
);

CREATE TABLE DIM_DATE (
    id_date INTEGER PRIMARY KEY,
    annee INTEGER NOT NULL,
    mois TEXT NOT NULL,
    jour INTEGER NOT NULL
);

CREATE TABLE DIM_GEO (
    id_geo INTEGER PRIMARY KEY,
    pays TEXT NOT NULL,
    code_pays TEXT NOT NULL
);

CREATE TABLE DIM_CATEGORIE (
    id_categorie INTEGER PRIMARY KEY,
    categorie TEXT NOT NULL
);

CREATE TABLE FACT_PERFORMANCE (
    id_fait INTEGER PRIMARY KEY,
    id_chaine INTEGER NOT NULL,
    id_date INTEGER NOT NULL,
    id_geo INTEGER NOT NULL,
    id_categorie INTEGER NOT NULL,
    abonnes INTEGER NOT NULL DEFAULT 0,
    vues INTEGER NOT NULL DEFAULT 0,
    uploads INTEGER NOT NULL DEFAULT 0,
    revenu_mensuel_max REAL NOT NULL DEFAULT 0,
    revenu_annuel_max REAL NOT NULL DEFAULT 0,
    FOREIGN KEY (id_chaine) REFERENCES DIM_CHAINE(id_chaine),
    FOREIGN KEY (id_date) REFERENCES DIM_DATE(id_date),
    FOREIGN KEY (id_geo) REFERENCES DIM_GEO(id_geo),
    FOREIGN KEY (id_categorie) REFERENCES DIM_CATEGORIE(id_categorie)
);
'''

conn.executescript(ddl)
print('tables creees avec DDL (PK + FK)')

dim_chaine.to_sql('DIM_CHAINE', conn, index=False, if_exists='append')
dim_date.to_sql('DIM_DATE', conn, index=False, if_exists='append')
dim_geo.to_sql('DIM_GEO', conn, index=False, if_exists='append')
dim_categorie.to_sql('DIM_CATEGORIE', conn, index=False, if_exists='append')
fact_performance.to_sql('FACT_PERFORMANCE', conn, index=False, if_exists='append')

print('donnees chargees dans youtube_dwh.db')

---
## Partie C - Data Warehouse SQL

### Schema en etoile

```
DIM_CHAINE ──┐
DIM_DATE   ──┼── FACT_PERFORMANCE (abonnes, vues, uploads, revenus)
DIM_GEO    ──┤
DIM_CATEGORIE┘
```

### Captures des tables (apercu)

In [ ]:
tables = ['DIM_CHAINE', 'DIM_DATE', 'DIM_GEO', 'DIM_CATEGORIE', 'FACT_PERFORMANCE']
for t in tables:
    n = pd.read_sql_query(f'SELECT COUNT(*) as n FROM {t}', conn).iloc[0]['n']
    print(f'\n=== {t} ({n} lignes) ===')
    display(pd.read_sql_query(f'SELECT * FROM {t} LIMIT 5', conn))

### Requetes analytiques

In [ ]:
q1 = '''
SELECT ch.youtuber, g.pays, f.abonnes
FROM FACT_PERFORMANCE f
JOIN DIM_CHAINE ch ON f.id_chaine = ch.id_chaine
JOIN DIM_GEO g ON f.id_geo = g.id_geo
ORDER BY f.abonnes DESC LIMIT 5
'''
print('Top 5 chaines :')
display(pd.read_sql_query(q1, conn))

In [ ]:
q2 = '''
SELECT c.categorie, SUM(f.abonnes) as total_abonnes, COUNT(*) as nb_chaines
FROM FACT_PERFORMANCE f
JOIN DIM_CATEGORIE c ON f.id_categorie = c.id_categorie
GROUP BY c.categorie ORDER BY total_abonnes DESC LIMIT 10
'''
print('Abonnes par categorie :')
display(pd.read_sql_query(q2, conn))

In [ ]:
q3 = '''
SELECT g.pays, COUNT(*) as nb_chaines, SUM(f.abonnes) as total_abonnes
FROM FACT_PERFORMANCE f
JOIN DIM_GEO g ON f.id_geo = g.id_geo
GROUP BY g.pays ORDER BY total_abonnes DESC LIMIT 5
'''
print('Top 5 pays :')
display(pd.read_sql_query(q3, conn))

In [ ]:
q4 = '''
SELECT d.annee, COUNT(*) as nb_chaines, SUM(f.abonnes) as total_abonnes
FROM FACT_PERFORMANCE f
JOIN DIM_DATE d ON f.id_date = d.id_date
GROUP BY d.annee ORDER BY d.annee
'''
result = pd.read_sql_query(q4, conn)
display(result.head(10))

plt.figure(figsize=(10, 5))
plt.plot(result['annee'], result['total_abonnes'], marker='o')
plt.title('Evolution des abonnes par annee')
plt.xlabel('Annee')
plt.ylabel('Total abonnes')
plt.show()

In [ ]:
q5 = '''
SELECT ch.type_chaine, AVG(f.abonnes) as abonnes_moy, AVG(f.revenu_annuel_max) as revenu_moy
FROM FACT_PERFORMANCE f
JOIN DIM_CHAINE ch ON f.id_chaine = ch.id_chaine
GROUP BY ch.type_chaine ORDER BY abonnes_moy DESC
'''
print('Performance par type de chaine :')
display(pd.read_sql_query(q5, conn))

### Operations SQL : UPDATE et DELETE

In [ ]:
print('UPDATE - avant :')
display(pd.read_sql_query("SELECT youtuber, pays FROM DIM_CHAINE ch JOIN FACT_PERFORMANCE f ON ch.id_chaine=f.id_chaine JOIN DIM_GEO g ON f.id_geo=g.id_geo WHERE youtuber='MrBeast'", conn))

conn.execute("UPDATE DIM_GEO SET pays='Canada' WHERE id_geo IN (SELECT f.id_geo FROM FACT_PERFORMANCE f JOIN DIM_CHAINE ch ON f.id_chaine=ch.id_chaine WHERE ch.youtuber='MrBeast')")
conn.commit()

print('UPDATE - apres :')
display(pd.read_sql_query("SELECT youtuber, pays FROM DIM_CHAINE ch JOIN FACT_PERFORMANCE f ON ch.id_chaine=f.id_chaine JOIN DIM_GEO g ON f.id_geo=g.id_geo WHERE youtuber='MrBeast'", conn))

In [ ]:
print('DELETE - avant :', pd.read_sql_query('SELECT COUNT(*) as n FROM FACT_PERFORMANCE', conn).iloc[0]['n'], 'lignes')

conn.execute('DELETE FROM FACT_PERFORMANCE WHERE abonnes < 1000000')
conn.commit()

print('DELETE - apres :', pd.read_sql_query('SELECT COUNT(*) as n FROM FACT_PERFORMANCE', conn).iloc[0]['n'], 'lignes')

In [ ]:
files.download('youtube_dwh.db')
conn.close()
print('fichier youtube_dwh.db telecharge')

---
## Partie D - Power BI (Bonus +3 pts)

1. Telecharger `youtube_dwh.db` depuis Colab
2. Ouvrir Power BI Desktop
3. Obtenir des donnees > Base de donnees SQLite > selectionner `youtube_dwh.db`
4. Charger les 5 tables (DIM_CHAINE, DIM_DATE, DIM_GEO, DIM_CATEGORIE, FACT_PERFORMANCE)
5. Creer les relations : FACT_PERFORMANCE.id_chaine -> DIM_CHAINE.id_chaine (et pareil pour date, geo, categorie)
6. Ajouter des visuels : barres (categorie), carte (pays), courbe (annee), KPI (total abonnes)
7. Faire une capture d ecran pour la presentation

---
## Preparation Q&A technique (8 pts)

**Qu est-ce qu un entrepot de donnees ?** Base centrale optimisee pour l analyse, structuree en schema en etoile.

**Qu est-ce que l ETL ?** Extract (lire CSV), Transform (nettoyer, creer dimensions/faits), Load (charger en base).

**Schema en etoile ?** 1 table de faits (mesures) + plusieurs dimensions (descriptions). Ici : FACT_PERFORMANCE + 4 DIM.

**PK et FK ?** PK = cle primaire (identifiant unique). FK = cle etrangere qui relie la table de faits aux dimensions.

**Pourquoi fillna Inconnu ?** Pour ne pas perdre de lignes et garder toutes les chaines dans le DWH.

**Outliers ?** Valeurs tres differentes de la mediane. Ex : T-Series avec 245M abonnes vs la majorite < 20M.